# 3D vision & point clouds (PointNet) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def conv2d(x,k,pad=0,stride=1,dilation=1):
    x=np.asarray(x,float); k=np.asarray(k,float)
    if pad: x=np.pad(x,pad)
    kh,kw=k.shape; dh=(kh-1)*dilation+1; dw=(kw-1)*dilation+1; out=[]
    for i in range(0,x.shape[0]-dh+1,stride):
        row=[]
        for j in range(0,x.shape[1]-dw+1,stride): row.append(float(np.sum(x[i:i+dh:dilation,j:j+dw:dilation]*k)))
        out.append(row)
    return np.array(out)
def iou(a,b):
    ax1,ay1,ax2,ay2=a; bx1,by1,bx2,by2=b
    ix1,iy1=max(ax1,bx1),max(ay1,by1); ix2,iy2=min(ax2,bx2),min(ay2,by2)
    inter=max(0,ix2-ix1)*max(0,iy2-iy1); union=(ax2-ax1)*(ay2-ay1)+(bx2-bx1)*(by2-by1)-inter
    return inter/union if union else 0
def softmax(z):
    z=np.asarray(z,float); e=np.exp(z-z.max()); return e/e.sum()
def show(a,title,cmap='viridis'):
    plt.figure(figsize=(4,3)); plt.imshow(a,cmap=cmap); plt.colorbar(); plt.title(title)
print('setup ok')

## 3D vision & point clouds (PointNet)
Tiny CPU-only synthetic arrays for 3D vision & point clouds (PointNet): no downloads, no pretrained models, and every cell ends with an assert.

In [ ]:
# 1) point cloud is an unordered set of xyz samples
pts=np.array([[0,0,0],[1,0,0],[0,1,0],[0,0,1]],float); fig=plt.figure(figsize=(4,3)); ax=fig.add_subplot(111,projection='3d'); ax.scatter(pts[:,0],pts[:,1],pts[:,2]); plt.title('point cloud')
assert pts.shape==(4,3)

In [ ]:
# 2) shared point features are max-pooled symmetrically
feat=np.array([[1,3],[2,1],[0,5.]]); g1=feat.max(0); g2=feat[[2,0,1]].max(0); plt.figure(figsize=(4,3)); plt.bar(['d0','d1'],g1); plt.title('PointNet max')
assert np.array_equal(g1,g2)

In [ ]:
# 3) a camera ray samples points through space
origin=np.array([0.,0.]); direction=np.array([1.,.5]); ts=np.array([0.,1.,2.,3.]); pts=origin+ts[:,None]*direction; plt.figure(figsize=(4,3)); plt.plot(pts[:,0],pts[:,1],'o-'); plt.title('ray samples')
assert np.allclose(pts[-1],[3,1.5])

In [ ]:
# 4) density becomes alpha opacity
sigma=np.array([.1,1.,3.]); alpha=1-np.exp(-sigma*.5); plt.figure(figsize=(4,3)); plt.bar(range(3),alpha); plt.title('alpha')
assert alpha[-1]>alpha[0]

In [ ]:
# 5) rendered color is weighted along the ray
colors=np.array([[1.,0,0],[0,1,0],[0,0,1]]); alpha=np.array([.2,.5,.1]); T=np.cumprod(np.r_[1,1-alpha[:-1]]); w=T*alpha; rgb=w@colors; plt.figure(figsize=(4,3)); plt.bar(['R','G','B'],rgb); plt.title('NeRF RGB')
assert np.allclose(rgb,[.2,.4,.04])